# Unzip cryoPANDA Data
Parallel unzipping of all `.zip` files found inside each sub-directory of the cryoPANDA root.

> **Requires `unzip` to be installed on your system:**
> ```bash
> sudo apt install unzip
> ```

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
CRYOPANDA_DIR    = 'PATH_TO_cryoPANDA_DIR'   # <-- set this
REMOVE_ZIP_FILES = True    # set False to keep .zip files after extraction
NUM_WORKERS      = 12      # parallel processes (capped to number of directories)

In [2]:
import os
import multiprocessing
import subprocess

## Unzip function

In [3]:
def unzip_directory(args):
    """
    Unzip all `.zip` files inside a single sub-directory.

    Parameters
    ----------
    args : tuple
        (di, directory_name, total, cryopanda_dir, remove_zip_files)
    """
    di, d, total, cryopanda_dir, remove_zip_files = args
    sub_dir = os.path.join(cryopanda_dir, d)

    try:
        zip_files = [f for f in os.listdir(sub_dir) if f.endswith('.zip')]
        if not zip_files:
            print(f'  [skip] No zip files in {d}')
            return

        for f in zip_files:
            full_path = os.path.join(sub_dir, f)
            subprocess.run(['unzip', '-o', f], check=True, cwd=sub_dir,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            if remove_zip_files:
                os.remove(full_path)
            print(f'  [{di+1}/{total}] {d}/{f} — done')

    except subprocess.CalledProcessError as exc:
        print(f'  [error] {d}: unzip failed — {exc}')
    except OSError as exc:
        print(f'  [error] {d}: OS error — {exc}')

## Run

In [ ]:
dir_list   = sorted(os.listdir(CRYOPANDA_DIR))
total      = len(dir_list)
n_workers  = min(NUM_WORKERS, total)

args = [(di, d, total, CRYOPANDA_DIR, REMOVE_ZIP_FILES) for di, d in enumerate(dir_list)]

print(f'Unzipping {total} directories using {n_workers} workers …')
with multiprocessing.Pool(processes=n_workers) as pool:
    pool.map(unzip_directory, args)
print('Done')

Unzipping 2 directories using 2 workers …
